# Multi-Object Detection and Persistent ID Tracking in Soccer Footage

## Project Overview

This project implements a computer vision pipeline for detecting and tracking multiple subjects in publicly available soccer footage.

The objective is to detect relevant subjects and maintain unique and persistent IDs across video frames while handling practical challenges such as player movement, partial occlusion, camera motion, scale changes, and visually similar subjects.

### Pipeline

1. YOLO-based object detection
2. ByteTrack baseline tracking
3. BoT-SORT with Re-ID and Global Motion Compensation (GMC)
4. Tracking performance comparison
5. Track persistence analysis
6. Player trajectory visualization
7. Final annotated video generation

## 2. Environment Setup and Imports

The project uses Python libraries for object detection, video processing, tracking data analysis, and trajectory visualization.

In [1]:
import cv2
import pandas as pd
import numpy as np

from pathlib import Path
from collections import defaultdict, deque

from ultralytics import YOLO

print("Libraries imported successfully.")

Libraries imported successfully.


## 3. Model and Video Configuration

The project uses a YOLO-based football player detection model and a selected soccer match video as the input source.

The model detects four classes: ball, goalkeeper, player, and referee.

In [2]:
MODEL_PATH = "models/football-player-detection.pt"
VIDEO_PATH = "data/soccer_match.mp4"

DETECTION_OUTPUT_DIR = "outputs/detection"
BYTETRACK_OUTPUT_DIR = "outputs/bytetrack"
BOTSORT_OUTPUT_DIR = "outputs/botsort"
TRACKING_DATA_DIR = "outputs/tracking_data"

print("Model path:", MODEL_PATH)
print("Video path:", VIDEO_PATH)

Model path: models/football-player-detection.pt
Video path: data/soccer_match.mp4


In [3]:
print("Model exists:", Path(MODEL_PATH).exists())
print("Video exists:", Path(VIDEO_PATH).exists())

Model exists: True
Video exists: True


## 4. YOLO-Based Object Detection

A YOLO-based football detection model is used to identify relevant subjects in the input soccer video.

The model provides detections for ball, goalkeeper, player, and referee classes. For the tracking pipeline, the relevant player-related detections are retained.

In [4]:
from ultralytics import YOLO

model = YOLO(MODEL_PATH)

print("Model classes:")
print(model.names)

Model classes:
{0: 'ball', 1: 'goalkeeper', 2: 'player', 3: 'referee'}


In [ ]:
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Selected device:", DEVICE)

results = model.predict(
    source=VIDEO_PATH,
    conf=0.25,
    imgsz=1280,
    device=DEVICE,
    save=True,
    project=DETECTION_OUTPUT_DIR,
    name="player_detection",
    exist_ok=True,
    verbose=True
)

print("\nDetection completed.")

## 5. Convert Detection Output to MP4

The YOLO detection output is initially saved as an AVI video. FFmpeg is used to convert the output into MP4 format for easier playback and visualization.

In [ ]:
import subprocess

detection_avi = Path(
    DETECTION_OUTPUT_DIR
) / "player_detection" / "soccer_match.avi"

detection_mp4 = Path(
    DETECTION_OUTPUT_DIR
) / "player_detection" / "soccer_match.mp4"

subprocess.run([
    "ffmpeg",
    "-y",
    "-i",
    str(detection_avi),
    "-c:v", "libx264",
    "-preset", "fast",
    "-pix_fmt", "yuv420p",
    "-an",
    str(detection_mp4)
], check=True)

print("Detection video converted to MP4:")
print(detection_mp4)

## 6. ByteTrack Baseline Tracking

ByteTrack is used as the baseline multi-object tracking method.

It associates detected objects across consecutive video frames and assigns a track ID to each detected subject. The resulting tracks provide a baseline for evaluating the more advanced BoT-SORT + Re-ID + GMC approach.

In [ ]:
bytetrack_results = model.track(
    source=VIDEO_PATH,
    tracker="bytetrack.yaml",
    conf=0.25,
    imgsz=1280,
    device=DEVICE,
    save=True,
    project=BYTETRACK_OUTPUT_DIR,
    name="baseline",
    exist_ok=True,
    verbose=True
)

print("\nByteTrack tracking completed.")

## 7. Convert ByteTrack Output to MP4

The ByteTrack tracking result is initially saved as an AVI video. FFmpeg is used to convert it to MP4 format for easier playback and comparison.

In [ ]:
bytetrack_avi = (
    Path(BYTETRACK_OUTPUT_DIR)
    / "baseline"
    / "soccer_match.avi"
)

bytetrack_mp4 = (
    Path(BYTETRACK_OUTPUT_DIR)
    / "baseline"
    / "soccer_match.mp4"
)

subprocess.run([
    "ffmpeg",
    "-y",
    "-i",
    str(bytetrack_avi),
    "-c:v", "libx264",
    "-preset", "fast",
    "-pix_fmt", "yuv420p",
    "-an",
    str(bytetrack_mp4)
], check=True)

print("ByteTrack video converted to MP4:")
print(bytetrack_mp4)

## 8. BoT-SORT + Re-ID + GMC

BoT-SORT is used as the advanced tracking approach.

Re-Identification (Re-ID) helps associate the same subject after temporary occlusion or appearance changes, while Global Motion Compensation (GMC) helps account for camera motion.

This configuration is evaluated against the ByteTrack baseline for improved ID persistence.

In [ ]:
BOTSORT_TRACKER = "configs/botsort_reid.yaml"

botsort_results = model.track(
    source=VIDEO_PATH,
    tracker=BOTSORT_TRACKER,
    conf=0.25,
    imgsz=1280,
    device=DEVICE,
    save=True,
    project=BOTSORT_OUTPUT_DIR,
    name="botsort_reid",
    exist_ok=True,
    verbose=True
)

print("BoT-SORT + Re-ID + GMC completed.")

## 9. Convert BoT-SORT Output to MP4

The BoT-SORT tracking result is initially saved as an AVI video. FFmpeg is used to convert the output into MP4 format for easier playback and visualization.

In [ ]:
botsort_avi = (
    Path(BOTSORT_OUTPUT_DIR)
    / "botsort_reid"
    / "soccer_match.avi"
)

botsort_mp4 = (
    Path(BOTSORT_OUTPUT_DIR)
    / "botsort_reid"
    / "soccer_match.mp4"
)

subprocess.run([
    "ffmpeg",
    "-y",
    "-i",
    str(botsort_avi),
    "-c:v", "libx264",
    "-preset", "fast",
    "-pix_fmt", "yuv420p",
    "-an",
    str(botsort_mp4)
], check=True)

print("BoT-SORT video converted to MP4:")
print(botsort_mp4)

## 10. Export Tracking Results to CSV

Tracking results are exported to CSV files containing frame-level information for each tracked object.

The CSV files store the frame number, track ID, class, confidence, and bounding-box coordinates. These records are later used for track persistence analysis and comparison between ByteTrack and BoT-SORT.

In [ ]:
import csv
from pathlib import Path
from ultralytics import YOLO


def collect_tracking_data(tracker, output_csv):
    tracking_model = YOLO(MODEL_PATH)

    rows = []

    results = tracking_model.track(
        source=VIDEO_PATH,
        tracker=tracker,
        conf=0.25,
        imgsz=1280,
        device=DEVICE,
        stream=True,
        persist=True,
        verbose=False
    )

    for frame_idx, result in enumerate(results):

        if result.boxes is None or result.boxes.id is None:
            continue

        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy().astype(int)
        classes = result.boxes.cls.cpu().numpy().astype(int)
        confs = result.boxes.conf.cpu().numpy()

        for box, track_id, cls, conf in zip(
            boxes, ids, classes, confs
        ):
            x1, y1, x2, y2 = box

            rows.append([
                frame_idx,
                track_id,
                cls,
                float(conf),
                float(x1),
                float(y1),
                float(x2),
                float(y2)
            ])

    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)

        writer.writerow([
            "frame",
            "track_id",
            "class_id",
            "confidence",
            "x1",
            "y1",
            "x2",
            "y2"
        ])

        writer.writerows(rows)

    print(f"Saved: {output_csv}")
    print(f"Tracking records: {len(rows)}")
    
    collect_tracking_data(
    "bytetrack.yaml",
    TRACKING_DATA_DIR + "/bytetrack.csv"
)
    
    collect_tracking_data(
    BOTSORT_TRACKER,
    TRACKING_DATA_DIR + "/botsort.csv"
)

## 11. Basic Tracking Statistics

The exported tracking CSV files are loaded and summarized to compare the tracking behavior of ByteTrack and BoT-SORT + Re-ID + GMC.

The analysis includes total tracking records, unique track IDs, frames containing detections, average active IDs per frame, and maximum active IDs.

In [ ]:
import pandas as pd

byte = pd.read_csv(
    TRACKING_DATA_DIR + "/bytetrack.csv"
)

bot = pd.read_csv(
    TRACKING_DATA_DIR + "/botsort.csv"
)


def summarize(df, name):
    print(f"\n===== {name} =====")
    print("Total tracking records :", len(df))
    print("Unique IDs             :", df["track_id"].nunique())
    print("Frames with detections :", df["frame"].nunique())

    active_per_frame = df.groupby("frame")["track_id"].nunique()

    print(
        "Average active IDs     :",
        round(active_per_frame.mean(), 2)
    )

    print(
        "Maximum active IDs     :",
        active_per_frame.max()
    )


summarize(byte, "ByteTrack")
summarize(bot, "BoT-SORT + Re-ID + GMC")

## 12. Track Persistence Analysis

Track persistence is analyzed by measuring the number of observations associated with each track ID and the duration for which each track remains active.

This helps evaluate track continuity and fragmentation for ByteTrack and BoT-SORT + Re-ID + GMC.

In [ ]:
def analyze_persistence(df, name):
    lifespan = (
        df.groupby("track_id")["frame"]
        .agg(["min", "max", "count"])
    )

    lifespan["duration"] = (
        lifespan["max"] - lifespan["min"] + 1
    )

    print(f"\n===== {name} =====")
    print("Number of tracks:", len(lifespan))
    print(
        "Average observations per ID:",
        round(lifespan["count"].mean(), 2)
    )
    print(
        "Average track duration:",
        round(lifespan["duration"].mean(), 2)
    )
    print(
        "Median track duration:",
        round(lifespan["duration"].median(), 2)
    )
    print(
        "Longest track:",
        lifespan["duration"].max()
    )


analyze_persistence(byte, "ByteTrack")
analyze_persistence(bot, "BoT-SORT + Re-ID + GMC")

## 13. ByteTrack vs BoT-SORT Comparison

The following comparison summarizes the observed tracking behavior of both approaches on the selected soccer sequence.

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Number of tracks",
        "Average observations per ID",
        "Average track duration",
        "Median track duration",
        "Longest track"
    ],
    "ByteTrack": [
        byte["track_id"].nunique(),
        round(
            byte.groupby("track_id").size().mean(), 2
        ),
        round(
            (
                byte.groupby("track_id")["frame"].max()
                - byte.groupby("track_id")["frame"].min()
                + 1
            ).mean(),
            2
        ),
        round(
            (
                byte.groupby("track_id")["frame"].max()
                - byte.groupby("track_id")["frame"].min()
                + 1
            ).median(),
            2
        ),
        (
            byte.groupby("track_id")["frame"].max()
            - byte.groupby("track_id")["frame"].min()
            + 1
        ).max()
    ],
    "BoT-SORT + Re-ID + GMC": [
        bot["track_id"].nunique(),
        round(
            bot.groupby("track_id").size().mean(), 2
        ),
        round(
            (
                bot.groupby("track_id")["frame"].max()
                - bot.groupby("track_id")["frame"].min()
                + 1
            ).mean(),
            2
        ),
        round(
            (
                bot.groupby("track_id")["frame"].max()
                - bot.groupby("track_id")["frame"].min()
                + 1
            ).median(),
            2
        ),
        (
            bot.groupby("track_id")["frame"].max()
            - bot.groupby("track_id")["frame"].min()
            + 1
        ).max()
    ]
})

comparison

### Observation

BoT-SORT + Re-ID + GMC produced fewer unique tracks and higher average observations per ID and average track duration than ByteTrack on the selected sequence.

These results indicate better track persistence and reduced track fragmentation in this experiment.

The comparison is based on tracking statistics from the selected video and does not represent a formal IDF1, HOTA, or ID-switch evaluation because ground-truth identity annotations are not available.

## 14. Final Submission Video

The final submission video combines BoT-SORT + Re-ID + GMC tracking with persistent track IDs, bounding boxes, player trajectories, active-track information, and a frame counter.

Only currently active tracks are visualized. A trajectory is reset when a track ID remains missing for more than the configured frame-gap threshold.

In [ ]:
import pandas as pd
import cv2
from collections import defaultdict, deque
from pathlib import Path

CSV_PATH = Path(TRACKING_DATA_DIR) / "botsort.csv"
INPUT_VIDEO_PATH = Path(VIDEO_PATH)
OUTPUT_PATH = Path("outputs/final_submission_video.mp4")

TRAJECTORY_LENGTH = 30
MAX_GAP = 2

df = pd.read_csv(CSV_PATH)

df["frame"] = df["frame"].astype(int)
df["track_id"] = df["track_id"].astype(int)

frame_data = {
    frame: group
    for frame, group in df.groupby("frame")
}

print("Tracking records:", len(df))
print("Unique IDs:", df["track_id"].nunique())

cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError(f"Unable to open video: {INPUT_VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS:", fps)
print("Resolution:", width, "x", height)
print("Total frames:", total_frames)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(OUTPUT_PATH),
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    raise RuntimeError(f"Unable to create output video: {OUTPUT_PATH}")

trajectories = defaultdict(
    lambda: deque(maxlen=TRAJECTORY_LENGTH)
)

last_seen = {}

frame_idx = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    current = frame_data.get(frame_idx)

    active_ids = set()

    if current is not None:

        for _, row in current.iterrows():

            track_id = int(row["track_id"])

            x1 = int(row["x1"])
            y1 = int(row["y1"])
            x2 = int(row["x2"])
            y2 = int(row["y2"])

            active_ids.add(track_id)

            if (
                track_id in last_seen
                and frame_idx - last_seen[track_id] > MAX_GAP
            ):
                trajectories[track_id].clear()

            last_seen[track_id] = frame_idx

            cx = int((x1 + x2) / 2)
            cy = int(y2)

            trajectories[track_id].append(
                (cx, cy)
            )

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            label = f"ID {track_id}"

            label_y1 = max(0, y1 - 30)

            cv2.rectangle(
                frame,
                (x1, label_y1),
                (x1 + 85, y1),
                (0, 255, 0),
                -1
            )

            cv2.putText(
                frame,
                label,
                (x1 + 5, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0, 0, 0),
                2
            )

    for track_id in active_ids:

        points = list(trajectories[track_id])

        if len(points) < 2:
            continue

        for i in range(1, len(points)):

            cv2.line(
                frame,
                points[i - 1],
                points[i],
                (0, 255, 255),
                2
            )

    cv2.rectangle(
        frame,
        (20, 20),
        (355, 100),
        (0, 0, 0),
        -1
    )

    cv2.putText(
        frame,
        "BoT-SORT + Re-ID + GMC",
        (35, 48),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Active Tracks: {len(active_ids)}",
        (35, 78),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Frame: {frame_idx + 1}/{total_frames}",
        (width - 270, 45),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        2
    )

    writer.write(frame)

    frame_idx += 1

    if frame_idx % 100 == 0:
        print(f"Processed {frame_idx}/{total_frames}")

cap.release()
writer.release()

print("FINAL VIDEO CREATED")

print(OUTPUT_PATH)